In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

In [2]:
INPUT_CSV = "NER_landslide_combined_final.csv"
MODEL_OUTPUT_PATH = "risk_model.pkl"
FEATURE_COLUMNS_OUTPUT_PATH = "model_feature_columns.pkl"

In [3]:
def encode_soil_data(df, soil_col="DOMSOI", min_count=20):
    """Same grouping + one-hot encoding logic used earlier."""
    df = df.copy()
 
    # Case 1: already one-hot encoded in a previous run (columns like
    # DOMSOI_Ao, DOMSOI_Bh already exist) - nothing to do, skip re-encoding.
    already_encoded_cols = [c for c in df.columns if c.startswith(f"{soil_col}_")]
    if already_encoded_cols:
        print(f"'{soil_col}' already one-hot encoded in this file "
              f"({len(already_encoded_cols)} columns found) - skipping re-encoding.")
        return df
 
    # Case 2: raw column not found at all under this exact name - fail loudly
    # with the actual column list, instead of a cryptic pandas KeyError.
    if soil_col not in df.columns:
        raise KeyError(
            f"Column '{soil_col}' not found, and no '{soil_col}_*' encoded "
            f"columns found either. Actual columns in this file:\n{list(df.columns)}\n"
            f"Check for a naming/case mismatch, or confirm which CSV you're loading."
        )
 
    # Case 3: normal case - raw text column present, encode it now
    df[soil_col] = df[soil_col].fillna("Unknown")
    value_counts = df[soil_col].value_counts()
    rare_categories = value_counts[value_counts < min_count].index.tolist()
    df[f"{soil_col}_grouped"] = df[soil_col].apply(
        lambda x: "Other" if x in rare_categories else x
    )
 
    dummies = pd.get_dummies(df[f"{soil_col}_grouped"], prefix=soil_col, dtype=int)
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(columns=[f"{soil_col}_grouped", soil_col])
    return df
 
 
def load_and_prepare_data():
    df = pd.read_csv(INPUT_CSV)
    df = encode_soil_data(df)
 
    # Columns that are identifiers/metadata/text - NOT model features
    non_feature_cols = [
        "event_date", "event_title", "event_description", "landslide_trigger",
        "country_name", "country_code","slope_deg" ,"admin_division_name",
        "longitude", "latitude", "location_accuracy_km", "FAOSOIL",
    ]
    non_feature_cols = [c for c in non_feature_cols if c in df.columns]
    df = df.drop(columns=non_feature_cols)
 
    X = df.drop(columns=["landslide_occurred"])
    y = df["landslide_occurred"]
 
    return X, y
 
 
def train_and_evaluate():
    X, y = load_and_prepare_data()
 
    print(f"Total rows: {len(X)}, Features: {list(X.columns)}")
    print(f"Class balance:\n{y.value_counts()}\n")
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
 
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",   # handles the ~3:1 negative:positive imbalance
        random_state=42,
    )
    model.fit(X_train, y_train)
 
    # ---- Evaluate ----
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
 
    print("Classification report (test set):")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC score: {roc_auc_score(y_test, y_proba):.3f}")
 
    # ---- Feature importance (useful for your PPT / Q&A defense) ----
    importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nFeature importance (which factors matter most):")
    print(importance)
 
    # ---- Save model + exact feature column order ----
    joblib.dump(model, MODEL_OUTPUT_PATH)
    joblib.dump(list(X.columns), FEATURE_COLUMNS_OUTPUT_PATH)
 
    print(f"\nSaved model to {MODEL_OUTPUT_PATH}")
    print(f"Saved feature column order to {FEATURE_COLUMNS_OUTPUT_PATH}")
    print("IMPORTANT: any future input to this model MUST have exactly these "
          "columns, in this exact order.")
 
    return model, X.columns
 
 
if __name__ == "__main__":
    train_and_evaluate()
 

'DOMSOI' already one-hot encoded in this file (11 columns found) - skipping re-encoding.
Total rows: 1398, Features: ['rainfall_1d', 'rainfall_3d', 'rainfall_7d', 'rainfall_14d', 'max_rainfall_7d', 'elevation_m', 'DOMSOI_Af', 'DOMSOI_Ao', 'DOMSOI_Bd', 'DOMSOI_Be', 'DOMSOI_Bh', 'DOMSOI_Gd', 'DOMSOI_Ge', 'DOMSOI_I', 'DOMSOI_Nd', 'DOMSOI_Other', 'DOMSOI_Rd']
Class balance:
landslide_occurred
0    1047
1     351
Name: count, dtype: int64

Classification report (test set):
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       210
           1       0.58      0.49      0.53        70

    accuracy                           0.78       280
   macro avg       0.71      0.68      0.69       280
weighted avg       0.77      0.78      0.78       280

ROC-AUC score: 0.765

Feature importance (which factors matter most):
rainfall_14d       0.241075
rainfall_3d        0.168369
rainfall_1d        0.143615
rainfall_7d        0.135575
elevation_m       